In [6]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.10
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [7]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [8]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [9]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [10]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [11]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[1.208 0.    0.   ]
 [0.    1.679 0.   ]
 [0.    0.    0.078]]


In [12]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,0.895,0.517,0.004,0.001,100000,3778,0.038,0.001,0.037,0.039
1,Infl,1.029,0.495,0.005,0.001,100000,5316,0.053,0.001,0.052,0.055
2,Rate,0.997,0.499,0.004,0.001,100000,4857,0.049,0.001,0.047,0.050


In [13]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.119,3.152,0.498,0.000,0.009,0.001,100000,6886,0.069,0.001,0.067,0.07,3.0,200,4
1,cov_identity,2.866,414.519,0.000,0.001,0.472,0.000,100000,100000,1.000,0.000,1.000,1.00,6.0,200,4


In [14]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.386,0.021,1.270,0.299,0.484,0.006,0.004,0.0,0.000,0.003,0.001,0.0,100000,6234,0.062,0.001,0.061,0.064
1,OutGap,x,-0.361,-0.091,0.269,-1.293,0.297,0.013,0.001,0.0,0.000,0.003,0.001,0.0,100000,23913,0.239,0.001,0.236,0.242
2,OutGap,r,-1.725,-0.053,2.261,-0.748,0.420,0.008,0.007,0.0,0.001,0.003,0.001,0.0,100000,11392,0.114,0.001,0.112,0.116
3,Infl,Pi,-0.644,-0.029,1.517,-0.410,0.473,0.006,0.005,0.0,0.000,0.003,0.001,0.0,100000,7066,0.071,0.001,0.069,0.072
4,Infl,x,0.034,0.009,0.322,0.124,0.492,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5520,0.055,0.001,0.054,0.057
5,Infl,r,-0.435,-0.009,2.704,-0.127,0.493,0.005,0.009,0.0,0.001,0.003,0.001,0.0,100000,5437,0.054,0.001,0.053,0.056
6,Rate,Pi,-0.072,-0.018,0.294,-0.251,0.484,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,6125,0.061,0.001,0.060,0.063
7,Rate,x,0.033,0.035,0.062,0.503,0.458,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,8401,0.084,0.001,0.082,0.086
8,Rate,r,-0.064,-0.006,0.523,-0.090,0.491,0.005,0.002,0.0,0.000,0.003,0.001,0.0,100000,5720,0.057,0.001,0.056,0.059


In [15]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-0.627,-0.051,0.846,-0.729,0.446,0.007,0.002,0.0,0.000,0.003,0.001,0.0,100000,8150,0.082,0.001,0.080,0.083
1,OutGap,x,-0.245,-0.093,0.178,-1.321,0.283,0.012,0.001,0.0,0.000,0.003,0.001,0.0,100000,21643,0.216,0.001,0.214,0.219
0,OutGap,r,-0.920,-0.030,2.158,-0.428,0.506,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,4361,0.044,0.001,0.042,0.045
5,Infl,Pi,-0.353,-0.023,1.010,-0.332,0.484,0.006,0.003,0.0,0.000,0.003,0.001,0.0,100000,6184,0.062,0.001,0.060,0.063
4,Infl,x,-0.040,-0.010,0.214,-0.141,0.499,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5078,0.051,0.001,0.049,0.052
3,Infl,r,-0.032,-0.000,2.578,-0.000,0.504,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,4699,0.047,0.001,0.046,0.048
8,Rate,Pi,0.026,0.008,0.196,0.116,0.499,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5035,0.050,0.001,0.049,0.052
7,Rate,x,0.019,0.029,0.041,0.405,0.476,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,6844,0.068,0.001,0.067,0.070
6,Rate,r,-0.089,-0.009,0.498,-0.133,0.499,0.005,0.002,0.0,0.000,0.003,0.001,0.0,100000,5165,0.052,0.001,0.050,0.053


In [25]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.754497,-1.368702,0.385795,0.385795,1.561518e-18,3.232038e-16,1.776940e-15,0.002663,0.002075,0.004079,0.004079,1.332071e-18,8.542956e-19,7.892080e-19
1,OutGap,x,-0.002398,-0.358930,-0.361328,-0.361328,-1.409621e-19,6.818369e-17,1.776940e-15,0.000573,0.000508,0.000892,0.000892,2.966685e-19,2.037690e-19,7.892080e-19
2,OutGap,r,-0.193246,-1.531500,-1.724746,-1.724746,-9.053608e-19,5.069833e-16,1.776940e-15,0.004837,0.004154,0.007344,0.007344,2.159505e-18,1.446764e-18,7.892080e-19
3,Infl,Pi,-0.059708,-0.584102,-0.643811,-0.643811,-1.761099e-17,3.691833e-16,1.776940e-15,0.001508,0.004663,0.004874,0.004874,1.534084e-18,9.967632e-19,7.892080e-19
4,Infl,x,0.010002,0.023501,0.033503,0.033503,1.223989e-20,7.745102e-17,1.776940e-15,0.000322,0.001009,0.001047,0.001047,3.215680e-19,2.083717e-19,7.892080e-19
5,Infl,r,-0.074270,-0.361034,-0.435304,-0.435304,-8.410271e-19,6.474127e-16,1.776940e-15,0.002714,0.008504,0.008823,0.008823,2.701516e-18,1.762589e-18,7.892080e-19
6,Rate,Pi,-0.001058,-0.070735,-0.071793,-0.071793,8.478356e-19,1.319778e-16,1.776940e-15,0.000325,0.000915,0.000949,0.000949,5.289965e-19,3.250560e-19,7.892080e-19
7,Rate,x,-0.000107,0.032898,0.032791,0.032791,2.786646e-20,2.817742e-17,1.776940e-15,0.000069,0.000205,0.000206,0.000206,1.136461e-19,7.053867e-20,7.892080e-19
8,Rate,r,-0.020042,-0.044371,-0.064412,-0.064412,1.074807e-18,2.409990e-16,1.776940e-15,0.000581,0.001726,0.001726,0.001726,9.689250e-19,5.983446e-19,7.892080e-19


In [26]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.826842,-2.453473,-0.626631,-0.626631,1.184861e-18,3.091859e-16,1.776940e-15,0.001753,0.001002,0.002380,0.002380,1.246924e-18,7.738635e-19,7.892080e-19
1,OutGap,x,0.285886,-0.530513,-0.244627,-0.244627,-2.576337e-19,6.311257e-17,1.776940e-15,0.000358,0.000328,0.000518,0.000518,2.603871e-19,1.672415e-19,7.892080e-19
2,OutGap,r,-1.190161,0.270371,-0.919790,-0.919790,5.052041e-19,4.661648e-16,1.776940e-15,0.005550,0.003465,0.006050,0.006050,1.947924e-18,1.273300e-18,7.892080e-19
3,Infl,Pi,-0.008598,-0.343997,-0.352595,-0.352595,-2.018669e-17,2.456649e-16,1.776940e-15,0.001007,0.003063,0.003204,0.003204,1.013938e-18,6.546968e-19,7.892080e-19
4,Infl,x,0.001164,-0.041293,-0.040128,-0.040128,-3.206744e-18,5.128858e-17,1.776940e-15,0.000213,0.000658,0.000682,0.000682,2.128750e-19,1.382503e-19,7.892080e-19
5,Infl,r,-0.043845,0.012200,-0.031644,-0.031644,1.343794e-17,6.149360e-16,1.776940e-15,0.002569,0.007831,0.008094,0.008094,2.552887e-18,1.654551e-18,7.892080e-19
6,Rate,Pi,0.000146,0.025411,0.025557,0.025557,4.321869e-19,8.788070e-17,1.776940e-15,0.000217,0.000588,0.000618,0.000618,3.518003e-19,2.157183e-19,7.892080e-19
7,Rate,x,0.000057,0.018476,0.018532,0.018532,1.609787e-19,1.866757e-17,1.776940e-15,0.000046,0.000132,0.000134,0.000134,7.509243e-20,4.641460e-20,7.892080e-19
8,Rate,r,-0.013253,-0.075298,-0.088551,-0.088551,-1.846725e-19,2.294361e-16,1.776940e-15,0.000552,0.001593,0.001598,0.001598,9.216648e-19,5.683765e-19,7.892080e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [16]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [17]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.305,-1294.303,-1125.797,337.011,0.0,0.0,0.127,0.051,0.179,0.0,100000,100000,1.0,0.0,1.0,1.0


In [18]:
res_mle

OptimizationResult(kind='mle', x=array([1.16703644]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.167036443030606), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1103.8767212013408), loglik=np.float64(-1103.8767212013408), logprior=np.float64(0.0), logpost=np.float64(-1103.8767212013408), nfev=14, nit=6, raw=  message: CONVERGENCE:

## Serial Autocorrelation Tests for the Augmented Model

In [19]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.962,0.361,0.007,0.001,100000,16569,0.166,0.001,0.163,0.168
1,Infl,3.929,0.189,0.011,0.001,100000,40850,0.408,0.002,0.405,0.412
2,Rate,1.000,0.498,0.004,0.001,100000,4954,0.050,0.001,0.048,0.051


In [20]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.119,3.152,0.498,0.000,0.009,0.001,100000,6886,0.069,0.001,0.067,0.07,3.0,200,4
1,cov_identity,2.866,414.519,0.000,0.001,0.472,0.000,100000,100000,1.000,0.000,1.000,1.00,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.096,2.855,0.524,0.0,0.008,0.001,100000,4822,0.048,0.001,0.047,0.050,3.0,200,4
1,cov_identity,0.464,72.955,0.001,0.0,0.123,0.000,100000,99748,0.997,0.000,0.997,0.998,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
